# Simple Classification
1. Loop through each spanned annotations and try to reverse find the source text that contains this annotation (positive chunk) - classify if a chunk contains a span
2. Randomly find chunks that do not have annotations (negative chunks)
3. Turn source text to vectors using Roberta
4. Train classifier
---
1. Load data, chunk, inspect
2. Build one spans table (contract_id, category_id, start, end) for the 10 categories
3. Label chunks per category
4. Check label counts per category
5. Split train/val by contract
6. Embed with RoBERTa
7. Train and evaluate


In [26]:
from utils import load_file_from_github, load_split_data, normalize_cuad

FROZEN_DIR = "task12b_outputs"

training_data = load_split_data("train", cuad=True)

raw_test = load_file_from_github("data/cuad/test.json")
test_data = normalize_cuad(raw_test["data"], cuad=True)

final_ids = [
    "governing_law",
    "renewal_term",
    "revenue_profit_sharing",
    "cap_on_liability",
    "uncapped_liability",
    "termination_for_convenience",
    "anti_assignment",
    "audit_rights",
    "license_grant",
    "exclusivity",
]
def get_final_10(data):
    # get all annotation sets in the final 10 categories
    sets = data.annotation_sets[data.annotation_sets.category_id.isin(final_ids)]

    # get all documents corresponding to the final annotation sets
    documents = data.documents[data.documents["contract_id"].isin(sets["contract_id"])]

    # get all spans in the final 10 categories
    spans = data.spans.merge(sets[["annotation_set_id", "category_id", "contract_id"]], on="annotation_set_id")

    return sets, documents, spans

train_sets, train_documents, train_spans = get_final_10(training_data)
test_sets, test_documents, test_spans = get_final_10(test_data)

train_spans.head()

,span_id,annotation_set_id,source_qa_id,answer_text,answer_start,answer_end,category_id,contract_id
0,contract_0001__renewal_term__span_000,contract_0001__renewal_term,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,If Distributor comp...,5543,5881,renewal_term,contract_0001
1,contract_0001__governing_law__span_000,contract_0001__governing_law,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,This Agreement is to be construed according to...,52061,52151,governing_law,contract_0001
2,contract_0001__exclusivity__span_000,contract_0001__exclusivity,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,The Company appoints the Distributor as an ex...,1854,2004,exclusivity,contract_0001
3,contract_0001__exclusivity__span_001,contract_0001__exclusivity,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Company hereby appoints ...,2112,2611,exclusivity,contract_0001
4,contract_0001__exclusivity__span_002,contract_0001__exclusivity,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,The Distributor shall not order or ...,12390,12503,exclusivity,contract_0001


# Chunk Document Context

Get training chunks from approved frozen chunks.

- Chunking by paragraph
- DistilRoBERTa tokenizer
- Chunk Size = 512 tokens
- Overlap = 64 tokens
- Result: 10,841 Chunks

In [27]:
train_chunks = load_file_from_github("notebooks/task10_outputs/chunks.parquet")
# add contract text to chunks
train_chunks["text"] = train_chunks.apply(
    lambda row: train_documents.loc[
        train_documents["contract_id"] == row["contract_id"], "context"
    ].iloc[0][row["char_start"]:row["char_end"]],
    axis=1
)
train_chunks.head()

,method,contract_id,context_group_id,chunk_id,chunk_number,char_start,char_end,num_tokens,chunk_text,chunk_hash,document_hash,text
0,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0000,0,0,2056,463,EXHIBIT 10.6\n\n ...,8d9734236056eaa8a3619bf2942ca9b27adadb920c52ff...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...,EXHIBIT 10.6\n\n ...
1,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0001,1,2076,3126,512,1.1 Grant and Acceptance. Company h...,0eb8801fa97ea183343a724bb6fbd0caab779ccfcdb942...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...,1.1 Grant and Acceptance. Company h...
2,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0002,2,2979,3717,359,such customers in the Market to those custom...,4e070f16a388ecff6151b6344f9ff838669ee88d9e2424...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...,such customers in the Market to those custom...
3,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0003,3,3737,4788,512,1.2 License. The Company hereby grants ...,fe8b1735b9cc0a5536b6771e0f47e30c7c80c93992179d...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...,1.2 License. The Company hereby grants ...
4,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0004,4,4658,5228,277,sublicense the Names or to do ...,c2b10a8c277aef96161b58f42a2068a32e017068e12f31...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...,sublicense the Names or to do ...


In [7]:
train_chunks["num_tokens"].max()

np.int64(512)

# Positives

Loop through each spanned annotations and try to reverse find the source text that contains this annotation (positive chunk) - classify if a chunk contains a span

In [28]:
import pandas as pd

def find_positives(chunks, spans):
    """
    Reverse find the chunks contains an annotation (positive chunk)
    :param chunks: a DataFrame of chunks with at least chunk_id and text categories
    :return: a DataFrame of (chunk_id, category_id) pairs
    """
    chunks_by_contract_idx = chunks.groupby("contract_id").indices
    start, end = chunks.char_start.values, chunks.char_end.values

    hits, pos_found = [], 0
    for row in spans.itertuples():
        chunk_idxes = chunks_by_contract_idx.get(row.contract_id)

        # if contract has no annotation sets
        if chunk_idxes is None:
            continue

        # matches chunk indices where span start and end are contained in the chunk
        matches = chunk_idxes[(start[chunk_idxes] <= row.answer_start)
                              & (row.answer_end <= end[chunk_idxes])]
        pos_found += len(matches) > 0

        # saves the chunk_id and their category of each match
        hits += [(row.category_id, chunks.chunk_id.iat[i]) for i in matches]

    positives = pd.DataFrame(hits, columns=["category_id", "chunk_id"]).drop_duplicates()
    return positives, pos_found

train_positives, pos_found = find_positives(train_chunks, train_spans)
print(f"{pos_found}/{len(train_spans)} spans matched a chunk")
train_positives.groupby("category_id").size().reset_index(name="count")

2990/3078 spans matched a chunk


,category_id,count
0,anti_assignment,313
1,audit_rights,259
2,cap_on_liability,333
3,exclusivity,203
4,governing_law,291
5,license_grant,386
6,renewal_term,127
7,revenue_profit_sharing,178
8,termination_for_convenience,143
9,uncapped_liability,91


# Negatives

Randomly find chunks that do not have annotations (negative chunks)

In [29]:
def get_chunk_categories(chunks, positives):
    """
    Pairs chunk_ids with each category it aligns with
    :param chunks: a DataFrame of chunks with at least chunk_id and text categories
    :param positives: a DataFrame of positive chunks with at least chunk_id and text categories
    :return: a DataFrame of (chunk_id, category_id) pairs
    """
    category_ids = positives["category_id"].unique().tolist()

    # cartesian product of chunk_id and category_id
    chunk_categories = pd.merge(chunks["chunk_id"], pd.Series(category_ids, name="category_id"), how="cross")

    # mark all tuples in positives as chunk contains category
    chunk_categories = chunk_categories.merge(
        positives.assign(is_contained=True),
        on=["chunk_id", "category_id"], how="left",
    )
    chunk_categories["is_contained"] = chunk_categories["is_contained"].fillna(False).astype(bool)

    return chunk_categories

train_chunk_categories = get_chunk_categories(train_chunks, train_positives)

# negatives are all tuples where is_contained == False
def find_negatives(chunk_categories):
    return chunk_categories.loc[~chunk_categories["is_contained"], ["chunk_id", "category_id"]]

train_negatives = find_negatives(train_chunk_categories)
train_negatives

,chunk_id,category_id
0,contract_0001__chunk_0000,renewal_term
1,contract_0001__chunk_0000,governing_law
3,contract_0001__chunk_0000,anti_assignment
4,contract_0001__chunk_0000,license_grant
5,contract_0001__chunk_0000,audit_rights
...,...,...
108405,contract_0408__chunk_0013,audit_rights
108406,contract_0408__chunk_0013,uncapped_liability
108407,contract_0408__chunk_0013,cap_on_liability
108408,contract_0408__chunk_0013,termination_for_convenience


In [30]:
train_negatives.groupby("category_id").size().reset_index(name="count")

,category_id,count
0,anti_assignment,10528
1,audit_rights,10582
2,cap_on_liability,10508
3,exclusivity,10638
4,governing_law,10550
5,license_grant,10455
6,renewal_term,10714
7,revenue_profit_sharing,10663
8,termination_for_convenience,10698
9,uncapped_liability,10750


Confirm that no information was lost during the positive/negative computations and split.

In [31]:
assert len(train_chunk_categories) == len(train_chunks) * len(train_positives["category_id"].unique().tolist())
assert train_chunk_categories["is_contained"].sum() == len(train_positives)
assert len(train_negatives) + len(train_positives) == len(train_chunk_categories)

print("Everything adds up!")

Everything adds up!


Freeze chunks for reuse (e.g. in Google Colab)

In [13]:
# freeze chunks for use in Google Colab
train_chunks[["chunk_id", "text"]].to_parquet(f"{FROZEN_DIR}/train_chunks.parquet")

In [32]:
import pandas as pd
train_chunks = pd.read_parquet(f"{FROZEN_DIR}/train_chunks.parquet")
train_chunks.head()

,chunk_id,text
0,contract_0001__chunk_0000,EXHIBIT 10.6\n\n ...
1,contract_0001__chunk_0001,1.1 Grant and Acceptance. Company h...
2,contract_0001__chunk_0002,such customers in the Market to those custom...
3,contract_0001__chunk_0003,1.2 License. The Company hereby grants ...
4,contract_0001__chunk_0004,sublicense the Names or to do ...


# Imbalance

In [8]:
imbalance = (
    train_chunk_categories.groupby("category_id")["is_contained"]
    .agg(n_pos="sum", n_total="size")
    .assign(
        n_neg=lambda d: d.n_total - d.n_pos,
        pos_rate=lambda d: d.n_pos / d.n_total,
        neg_per_pos=lambda d: (d.n_total - d.n_pos) / d.n_pos,
    )
    .sort_values("pos_rate")
)
imbalance.round(4)

,n_pos,n_total,n_neg,pos_rate,neg_per_pos
category_id,,,,,
uncapped_liability,91,10841,10750,0.0084,118.1319
renewal_term,127,10841,10714,0.0117,84.3622
termination_for_convenience,143,10841,10698,0.0132,74.8112
revenue_profit_sharing,178,10841,10663,0.0164,59.9045
exclusivity,203,10841,10638,0.0187,52.4039
audit_rights,259,10841,10582,0.0239,40.8571
governing_law,291,10841,10550,0.0268,36.2543
anti_assignment,313,10841,10528,0.0289,33.6358
cap_on_liability,333,10841,10508,0.0307,31.5556


In [9]:
imbalance["pos_rate"].sum()

np.float64(0.21437136795498574)

# Vector Embeddings
Turn source text to vectors using Roberta

In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer

def create_embeddings(chunks, prefix):
    model = SentenceTransformer("sentence-transformers/all-distilroberta-v1")

    chunks = chunks.reset_index(drop=True)      # row position == matrix position
    texts = chunks["text"].tolist()

    embeddings = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,   # unit-length, so dot product = cosine sim
    )                                # float32 numpy array, shape (len(texts), 768)

    assert embeddings.shape == (len(chunks), model.get_sentence_embedding_dimension())

    np.savez(f"{FROZEN_DIR}/{prefix}_embeddings.npz",
             chunk_ids=chunks["chunk_id"].to_numpy().astype(str),   # dtype '<U..', not object
             embeddings=embeddings)

# Example: create_embeddings(train_chunks, "train")
# Recommended to run on Google Colab to take advantage of its T4 GPU

/Users/amiebinan/Desktop/Internships/Accenture-1O-contract-review-challenge/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Import vector embeddings from Google Colab notebook

In [33]:
import numpy as np
def load_embeddings(prefix, chunks):
    d = np.load(f"{FROZEN_DIR}/{prefix}_embeddings.npz")
    embeddings, ids = d["embeddings"], d["chunk_ids"]

    chunks = chunks.reset_index(drop=True)

    # confirm that embeddings rows match the chunk_id rows
    assert len(embeddings) == len(chunks)
    assert (ids == chunks["chunk_id"].to_numpy().astype(str)).all()

    chunks["embedding"] = list(embeddings)
    return chunks, embeddings

train_chunks, train_embeddings = load_embeddings("train", train_chunks)
train_chunks.head()

,chunk_id,text,embedding
0,contract_0001__chunk_0000,EXHIBIT 10.6\n\n ...,"[-0.0101947775, -0.004966498, -0.0014591962, -..."
1,contract_0001__chunk_0001,1.1 Grant and Acceptance. Company h...,"[0.010386001, 0.00900911, 0.0069864546, -0.006..."
2,contract_0001__chunk_0002,such customers in the Market to those custom...,"[-0.011106098, -0.013900779, 0.015085381, 0.01..."
3,contract_0001__chunk_0003,1.2 License. The Company hereby grants ...,"[-0.022413723, 0.049107153, -0.018923167, -0.0..."
4,contract_0001__chunk_0004,sublicense the Names or to do ...,"[-0.007902839, 0.03549104, 0.008908994, -0.003..."


# Testing

We are using an 64/20 split of training and test data, ommitting validation in the meantime.

In [34]:
from chunking_helper import create_paragraph_chunks
from transformers import AutoTokenizer

MODEL_ID = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

test_chunks = create_paragraph_chunks(test_documents, tokenizer, chunk_size=512)

test_chunks["text"] = test_chunks.apply(
    lambda row: test_documents.loc[
        test_documents["contract_id"] == row["contract_id"], "context"
    ].iloc[0][row["char_start"]:row["char_end"]],
    axis=1
)

test_positives, _ = find_positives(test_chunks, test_spans)
test_chunk_categories = get_chunk_categories(test_chunks, test_positives)
test_negatives = find_negatives(test_chunk_categories)

# freeze chunks for use in Google Colab
test_chunks[["chunk_id", "text"]].to_parquet(f"{FROZEN_DIR}/test_chunks.parquet")

# create_embeddings(test_chunks, "test")
test_chunks, test_embeddings = load_embeddings("test", test_chunks)

print("Test data loaded successfully!")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (573 > 512). Running this sequence through the model will result in indexing errors


Test data loaded successfully!


# Logistic Regression

A classifier for all provided categories by instantiating an individual LogisticRegression model for each category.


In [35]:
from sklearn.linear_model import LogisticRegression
import pandas as pd

class LogisticRegressionByCategory:
    """One binary logistic regression per category, on chunk embeddings."""

    def __init__(self, categories, **lr_kwargs):
        """
        :param categories: ids of all categories to be classified
        :param lr_kwargs:  arguments for scikit-learn's logistic regression
        """
        self.lr_kwargs = {"class_weight": "balanced", **lr_kwargs}
        self.categories = categories
        self.models = {}

    def fit(self, X, y):
        """
        Trains a Logistic Regression model for each of the categories
        :param X: np.array containing chunk embeddings for training data
        :param y: DataFrame aligned with X, providing 0/1 per category
        :return:
        """
        # y: DataFrame, one 0/1 column per category, rows aligned with X
        self.models = {
            cat: LogisticRegression(**self.lr_kwargs).fit(X, y[cat])
            for cat in self.categories
        }
        return self

    def predict_proba(self, X):
        """
        Predicts probability of a chunk being a part of a category, for every category
        :param X: a np.array containing chunk embeddings for test data
        :return:
        """
        return {cat: m.predict_proba(X)[:, 1] for cat, m in self.models.items()}

    def predict(self, X, threshold=0.5, index=None):
        """
        :param X: np.array of chunk embeddings
        :param threshold: float, or dict of category -> float
        :param index: chunk ids for the rows of X (e.g. test_chunks["chunk_id"])
        :return: DataFrame of 0/1 predictions, one column per category
        """
        proba = pd.DataFrame(self.predict_proba(X), index=index)
        if isinstance(threshold, dict):
            return pd.DataFrame(
                {cat: (proba[cat] >= threshold.get(cat, 0.5)).astype(int) for cat in proba.columns}
            )
        return (proba >= threshold).astype(int)

Train and test the classifier...

In [48]:
from logistic_regression_by_category import LogisticRegressionByCategory

X_train = train_embeddings
y_train = (train_chunk_categories
           .pivot(index="chunk_id", columns="category_id", values="is_contained")
           .reindex(train_chunks["chunk_id"])
           .fillna(0)
           .astype(int)
           )

X_test = test_embeddings
y_test = (test_chunk_categories
           .pivot(index="chunk_id", columns="category_id", values="is_contained")
           .reindex(test_chunks["chunk_id"])
           .fillna(0)
           .astype(int)
           )

clf = LogisticRegressionByCategory(final_ids, random_state=42)
clf.fit(X_train, y_train)

In [49]:
predictions = clf.predict(X_test, threshold=0.5, index=test_chunks["chunk_id"])
predictions

,governing_law,renewal_term,revenue_profit_sharing,cap_on_liability,uncapped_liability,termination_for_convenience,anti_assignment,audit_rights,license_grant,exclusivity
chunk_id,,,,,,,,,,
contract_0001__chunk_0000,1,0,0,0,0,0,1,1,0,0
contract_0001__chunk_0001,0,0,0,0,0,0,0,0,0,0
contract_0001__chunk_0002,0,0,0,0,0,0,0,0,0,0
contract_0001__chunk_0003,0,0,0,0,0,0,0,1,0,0
contract_0001__chunk_0004,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
contract_0102__chunk_0009,0,0,0,0,0,0,0,0,0,0
contract_0102__chunk_0010,1,0,0,0,0,0,1,0,0,0
contract_0102__chunk_0011,0,0,0,0,0,0,0,0,0,0


In [50]:
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate(y_true, y_pred):
    """Both DataFrames: same index/columns, 0/1 values. Returns per-category metrics."""
    rows = []
    for cat in y_true.columns:
        t, p = y_true[cat].to_numpy(), y_pred[cat].to_numpy()
        rows.append({
            "category": cat,
            "support": t.sum(),
            "precision": precision_score(t, p, zero_division=0),
            "recall": recall_score(t, p, zero_division=0),
            "f1": f1_score(t, p, zero_division=0),
        })
    return pd.DataFrame(rows).set_index("category")

y_pred = clf.predict(X_test)
results = evaluate(y_test[final_ids], y_pred)
results[results["support"] > 0][["recall", "f1"]].mean()   # macro, ignoring empty categories

recall    0.863680
f1        0.301672
dtype: float64